# Baseline Linear Regression

training linear regression in blocks using baseline features + traficom features

## 1. Imports

In [142]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load data

In [143]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [144]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

## 3. Quick data checks

In [145]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 78)
Validation shape: (1689, 78)


In [146]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 78 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   product_id                      7954 non-null   int64  
 1   part_name                       7954 non-null   object 
 2   price                           7954 non-null   float64
 3   quality_grade                   7954 non-null   object 
 4   oem_number                      7954 non-null   object 
 5   mileage                         7190 non-null   float64
 6   brand                           7954 non-null   object 
 7   model                           7954 non-null   object 
 8   category                        7954 non-null   object 
 9   subcategory                     7954 non-null   object 
 10  scrape_date                     7954 non-null   object 
 11  year_start                      7954 non-null   int64  
 12  year_end               

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [147]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column in train_df.columns

]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_date": "Observed listing start within the crawl window; improved validation when included.",
    "last_seen_date": "Observed listing end within the crawl window; improved validation when included.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

recommended_excluded_features = set(recommended_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in recommended_excluded_features
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nNumber of recommended model features:", len(recommended_model_features))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,first_seen_date,missing_from_previous_manual_all,Observed listing start within the crawl window...
2,last_seen_date,missing_from_previous_manual_all,Observed listing end within the crawl window; ...
3,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
4,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
5,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
6,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
7,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
8,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...


## 5. Select baseline feature set

In [148]:
features_baseline_only = baseline_features

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 13


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status']

## 6. Build X and y

In [149]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [150]:
y_train_log = np.log(y_train)

y_train_log.head()

0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [151]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

## 8. Detect column types

In [152]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [153]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 9. Build preprocessing and model pipeline

In [154]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [155]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

## 10. Train baseline linear regression

In [156]:
baseline_model.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 11. Predict and convert back to euro scale

In [157]:
validation_pred_log = baseline_model.predict(X_validation)

In [158]:
validation_pred = np.exp(validation_pred_log)

## 12. Preview Baseline Predictions


In [159]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,172.063929
1,473.6,429.782357
2,142.1,138.763147
3,82.9,125.889618
4,177.6,95.641515


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [160]:
features_baseline_plus_traficom = baseline_features + traficom_features

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 26


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [161]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [162]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [163]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Build a fresh preprocessing and model pipeline

In [164]:
numeric_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [165]:
preprocessor_traficom = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_traficom, numeric_features_traficom),
    ("cat", categorical_pipeline_traficom, categorical_features_traficom),
])

linear_regression_traficom = Pipeline(steps=[
    ("preprocessor", preprocessor_traficom),
    ("model", LinearRegression()),
])

## 17. Train baseline + Traficom model

In [166]:
linear_regression_traficom.fit(X_train_traficom, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 18. Predict on validation and convert back to euro scale

In [167]:
validation_pred_log_traficom = linear_regression_traficom.predict(X_validation_traficom)

In [168]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Preview Baseline + Traficom Predictions

In [169]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,172.068145
1,473.6,429.803926
2,142.1,138.755326
3,82.9,125.264134
4,177.6,95.650639


In [170]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,248.662168
std,567.153248,532.325723
min,5.900000,8.032024
25%,59.000000,55.997836
50%,100.600000,98.103448
75%,236.000000,235.747427
max,4284.000000,4087.614434


## 20. All Recommended Features

This section trains the linear model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.

In [171]:
features_all = recommended_model_features

print("Number of recommended model features:", len(features_all))
features_all

Number of recommended model features: 71


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [172]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [173]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [174]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Build a fresh preprocessing and model pipeline

In [175]:
numeric_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])

In [176]:
preprocessor_all = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_all, numeric_features_all),
    ("cat", categorical_pipeline_all, categorical_features_all),
])

linear_regression_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all),
    ("model", LinearRegression()),
])

## 24. Train baseline + Traficom + registry lifecycle + listing dynamics model

In [177]:
linear_regression_all.fit(X_train_all, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 25. Predict on validation and convert back to euro scale

In [178]:
validation_pred_log_all = linear_regression_all.predict(X_validation_all)

In [179]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Preview Recommended-Feature Predictions

In [180]:
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,178.772975
1,473.6,441.045364
2,142.1,142.180837
3,82.9,112.323848
4,177.6,97.170501


In [181]:
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,252.367857
std,567.153248,561.573677
min,5.900000,7.901008
25%,59.000000,54.642618
50%,100.600000,99.089304
75%,236.000000,229.271523
max,4284.000000,6535.068196


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing linear regression model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.

In [182]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",178.772975,1.172975,1.375870
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",441.045364,32.554636,1059.804341
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",142.180837,0.080837,0.006535
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",112.323848,29.423848,865.762824
4,177.6,airbag,all,"Side airbags - , e-(Right)",97.170501,80.429499,6468.904367


In [183]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below show the ten strongest and ten weakest groups, filtered to groups with at least 20 validation observations.

In [184]:
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,8.43,6.60,11.42,59.1%,95.0%,100.0%
1,electric / transmitter / databox / sensor,404,13.78,6.22,41.24,65.8%,82.9%,97.8%
2,vehicle exterior / suspension,294,17.69,9.81,31.83,50.7%,79.9%,93.9%
3,gear box / drive axle / middle axle,241,40.02,21.99,89.09,33.2%,45.2%,84.2%
4,airbag,106,40.56,20.16,60.06,31.1%,49.1%,69.8%
5,brakes,214,45.63,21.30,91.65,36.0%,48.1%,80.4%
6,engine,249,109.87,10.30,389.53,49.0%,71.1%,79.5%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,109.87,10.30,389.53,49.0%,71.1%,79.5%
5,brakes,214,45.63,21.30,91.65,36.0%,48.1%,80.4%
4,airbag,106,40.56,20.16,60.06,31.1%,49.1%,69.8%
3,gear box / drive axle / middle axle,241,40.02,21.99,89.09,33.2%,45.2%,84.2%
2,vehicle exterior / suspension,294,17.69,9.81,31.83,50.7%,79.9%,93.9%
1,electric / transmitter / databox / sensor,404,13.78,6.22,41.24,65.8%,82.9%,97.8%
0,fuel,181,8.43,6.60,11.42,59.1%,95.0%,100.0%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,217.46,130.60,335.08,4.8%,4.8%,28.6%
22,passenger airbag,25,79.82,48.17,97.49,8.0%,8.0%,52.0%
21,adaptiv farthållare sensor,24,26.22,27.62,26.52,0.0%,16.7%,100.0%
20,right rear,49,25.26,21.60,33.95,18.4%,40.8%,95.9%
19,abs hydraulic pump,36,23.56,24.57,25.07,0.0%,33.3%,100.0%
18,left front,39,20.96,13.24,29.29,35.9%,51.3%,84.6%
17,all,153,20.96,11.57,33.49,48.4%,71.9%,88.2%
16,right front,48,18.42,11.23,24.27,45.8%,56.2%,97.9%
15,left rear,50,16.68,23.13,21.20,46.0%,46.0%,100.0%
14,deliverer,23,16.37,14.28,17.29,0.0%,91.3%,100.0%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,4.91,4.12,5.57,90.0%,100.0%,100.0%
1,Brake Caliper -(Left front),20,5.94,1.44,16.62,95.0%,95.0%,95.0%
2,Trailing link rear -(Left),20,8.81,8.06,9.20,70.0%,100.0%,100.0%
3,Wheel bearing spindle shaft -(Right rear),20,21.48,19.40,24.00,10.0%,55.0%,100.0%
4,Drive shaft -(Right front),23,26.40,25.39,30.77,21.7%,30.4%,91.3%
5,Wheel bearing spindle shaft -(Left rear),21,27.91,24.59,29.16,0.0%,23.8%,100.0%
6,ABS Hydraulic pump -,24,28.40,26.80,29.08,0.0%,0.0%,100.0%
7,Drive shaft -(Left front),32,28.61,24.49,33.49,3.1%,34.4%,81.2%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Drive shaft -(Left front),32,28.61,24.49,33.49,3.1%,34.4%,81.2%
6,ABS Hydraulic pump -,24,28.40,26.80,29.08,0.0%,0.0%,100.0%
5,Wheel bearing spindle shaft -(Left rear),21,27.91,24.59,29.16,0.0%,23.8%,100.0%
4,Drive shaft -(Right front),23,26.40,25.39,30.77,21.7%,30.4%,91.3%
3,Wheel bearing spindle shaft -(Right rear),20,21.48,19.40,24.00,10.0%,55.0%,100.0%
2,Trailing link rear -(Left),20,8.81,8.06,9.20,70.0%,100.0%,100.0%
1,Brake Caliper -(Left front),20,5.94,1.44,16.62,95.0%,95.0%,95.0%
0,Distributors Vacuum regulator -,20,4.91,4.12,5.57,90.0%,100.0%,100.0%


## 27. Select The Best Linear Regression Feature Set

Train every candidate linear regression first, then validate them together once at the end so the feature-set comparison stays in one place.

In [185]:
excluded_features = recommended_excluded_features

feature_sets = {
    "baseline only": baseline_features,
    "baseline + traficom_core": baseline_features + traficom_features,
    "baseline + traficom_core + registry_lifecycle": (
        baseline_features + traficom_features + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        baseline_features
        + traficom_features
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "previous manual all-features set": manual_all_feature_groups,
    "all recommended features": recommended_model_features,
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [186]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


def build_linear_regression_pipeline(X_train_current):
    numeric_features_current = X_train_current.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()
    categorical_features_current = X_train_current.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
    ])

    preprocessor_current = ColumnTransformer(transformers=[
        ("num", numeric_pipeline_current, numeric_features_current),
        ("cat", categorical_pipeline_current, categorical_features_current),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor_current),
        ("model", LinearRegression()),
    ])

In [187]:
for model_name, feature_list in feature_sets.items():
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    model_current = build_linear_regression_pipeline(X_train_current)
    model_current.fit(X_train_current, y_train_log)

    validation_pred_log_current = model_current.predict(X_validation_current)
    validation_pred_current = np.exp(validation_pred_log_current)
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 28. Validate The Best Linear Regression Model

Now that every candidate model has been trained, validate them once, choose the best feature set, and inspect only that winner.

In [188]:
if "best_model_validation_df" not in globals():
    if "best_experiment_name" not in globals():
        if "experiment_results" not in globals() or len(experiment_results) == 0:
            raise NameError(
                "best_experiment_name is not defined. Run the feature-set comparison section first."
            )

        experiment_results_df = pd.DataFrame(experiment_results)
        best_experiment_name = experiment_results_df.sort_values(
            ["validation_MAE", "validation_RMSE"]
        ).iloc[0]["experiment"]

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,237,34.2,40.6,6.35
1,"(5.899, 47.4]",underpredicted,112,38.2,33.1,5.13
2,"(47.4, 82.6]",overpredicted,175,61.0,67.2,6.16
3,"(47.4, 82.6]",underpredicted,160,61.3,53.6,7.69
4,"(82.6, 154.06]",overpredicted,176,114.1,128.2,14.08
5,"(82.6, 154.06]",underpredicted,153,105.1,90.7,14.40
6,"(154.06, 248.42]",overpredicted,178,199.5,227.0,27.54
7,"(154.06, 248.42]",underpredicted,160,211.4,185.5,25.88
8,"(248.42, 4284.0]",overpredicted,130,1016.7,1142.0,125.32
9,"(248.42, 4284.0]",underpredicted,208,800.8,661.6,139.23


In [189]:
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,5.96,8.81,19.8%,-1.67
1,"(47.4, 82.6]",335,47.6,82.6,61.1,6.89,10.66,11.2%,-0.31
2,"(82.6, 154.06]",329,82.8,153.9,109.9,14.23,22.49,12.6%,-0.68
3,"(154.06, 248.42]",338,154.1,248.3,205.1,26.75,43.21,13.7%,-1.34
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,133.88,352.59,13.9%,+19.99


In [190]:
experiment_results_df = pd.DataFrame(experiment_results)
baseline_row = experiment_results_df.loc[
    experiment_results_df["experiment"] == "baseline only"
].iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

display_columns = [
    "experiment",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
]

best_experiment_name = experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]

print(f"Best experiment by MAE: {best_experiment_name}")

experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
})

Best experiment by MAE: all recommended features


,experiment,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank
5,all recommended features,71,37.5153,-5.9240,1,159.3422,+4.5925,5
4,previous manual all-features set,68,38.6117,-4.8276,2,169.1225,+14.3727,6
3,baseline + traficom_core + registry_lifecycle + traficom_extended,62,43.3804,-0.0588,3,154.6992,-0.0505,1
2,baseline + traficom_core + registry_lifecycle,40,43.4145,-0.0248,4,154.7407,-0.0091,2
1,baseline + traficom_core,26,43.4170,-0.0223,5,154.7455,-0.0042,3
0,baseline only,13,43.4393,+0.0000,6,154.7497,+0.0000,4
